<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  
  <b>Module 2.1</b>  
  <span style="color:#9ca3af;">Document Loaders in LangChain</span>
</div>

LangChain provides loaders for virtually every document type. Each loader returns a list of `Document` objects with `.page_content` and `.metadata`.

| Loader | Source |
|---|---|
| `PyPDFLoader` | PDF files |
| `PDFMinerLoader` | PDFs (precise layout) |
| `PDFPlumberLoader` | PDFs with tables |
| `UnstructuredPDFLoader` | Complex PDFs |
| `WebBaseLoader` | Web pages |
| `RecursiveUrlLoader` | Entire websites |
| `CSVLoader` | CSV files |
| `JSONLoader` | JSON / JSONL |
| `TextLoader` | Plain text |

In [24]:
# !pip install langchain langchain-community pypdf pdfminer.six pdfplumber
# !pip install unstructured beautifulsoup4 requests

# ── 1. PDF Loaders ────────────────────────────────────────────────────────────
from langchain_community.document_loaders import (
    PyPDFLoader,
    PDFMinerLoader,
    PDFPlumberLoader,
)

def load_and_inspect(loader_class, path, **kwargs):
    loader = loader_class(path, **kwargs)
    docs   = loader.load()
    print(f"  Loader : {loader_class.__name__}")
    print(f"  Pages  : {len(docs)}")
    print(f"  Sample : {docs[0].page_content[:200].strip()}\n")
    return docs

# Data directory path
PDF_PATH = "../data/sample.pdf"

print("Loading PDF variants...")
try:
    load_and_inspect(PyPDFLoader, PDF_PATH)
    # load_and_inspect(PDFMinerLoader, PDF_PATH)
    # load_and_inspect(PDFPlumberLoader, PDF_PATH)
except Exception as e:
    print(f"  Error loading PDF: {e}")

Loading PDF variants...
  Loader : PyPDFLoader
  Pages  : 2
  Sample : Advanced RAG Systems: Technical Specifications
1. Executive Summary
This document outlines the architecture for a next-generation Retrieval-Augmented
Generation (RAG) system. The goal is to maximize r



In [25]:
# ── 2. Web Loaders ────────────────────────────────────────────────────────────
from langchain_community.document_loaders import WebBaseLoader
import bs4

# Single page
url = "https://python.langchain.com/docs/introduction/"
loader = WebBaseLoader(
    web_paths=[url],
    bs_kwargs={"parse_only": bs4.SoupStrainer(class_=("theme-doc-markdown",))}
)
try:
    web_docs = loader.load()
    print(f"WebBaseLoader — chars: {len(web_docs[0].page_content)}")
    print(web_docs[0].page_content[:300])
except Exception as e:
    print(f"Error fetching web page: {e}")

WebBaseLoader — chars: 0



In [26]:
# ── 3. CSV Loader ─────────────────────────────────────────────────────────────
from langchain_community.document_loaders import CSVLoader

csv_path = "../data/sample.csv"

loader = CSVLoader(file_path=csv_path, csv_args={"delimiter": ","})
csv_docs = loader.load()

print(f"CSV rows loaded: {len(csv_docs)}")
for d in csv_docs[:2]:
    print(" ", d.page_content.replace('\n', ' | '))

CSV rows loaded: 10
  id: 1 | category: Hardware | title: Server Maintenance | content: Quarterly server maintenance involves checking logs and clearing cache. | priority: High
  id: 2 | category: Software | title: CI/CD Pipeline | content: Automated testing should run on every pull request to ensure stability. | priority: Medium


In [27]:
# ── 4. JSON Loader ────────────────────────────────────────────────────────────
# Note: LangChain's JSONLoader requires 'jq', which is hard to install on Windows.
# We use a robust manual implementation for consistency.
import json
from langchain_core.documents import Document

json_path = "../data/sample.json"

def load_json_manual(path, content_key="content"):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return [Document(page_content=item[content_key], 
                     metadata={"source": path, "title": item.get("title", "")}) 
            for item in data]

json_docs = load_json_manual(json_path)
print(f"JSON items loaded: {len(json_docs)}")
for d in json_docs[:2]:
    print(" ", d.page_content)

JSON items loaded: 3
  Use LangGraph Cloud or a Docker container running Uvicorn.
  For general use, all-MiniLM-L6-v2 is fast and free. For high precision, text-embedding-3-large is recommended.


In [28]:
# ── 5. Text Loader ────────────────────────────────────────────────────────────
from langchain_community.document_loaders import TextLoader

txt_path = "../data/sample.txt"
loader = TextLoader(txt_path)
text_docs = loader.load()

print(f"Text documents loaded: {len(text_docs)}")
print(text_docs[0].page_content[:200])

Text documents loaded: 1
# Deep Dive into Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) is a breakthrough architectural pattern in the field of Large Language Models (LLMs). While LLMs like GPT-4,


## Hands-on Exercise
Load documents from 5 different sources and inspect their metadata and content. Try loading your own PDF, a company webpage, a CSV of product data, a JSON FAQ list, and a plain text readme.